<a href="https://colab.research.google.com/github/Laura-loaiza/Workshop-001/blob/main/ETL_Workshop01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop 1 - ETL de Candidatos

Objetivo: Con el CSV de candidatos, hacer el proceso de extracción y transformarlos haciendole limpieza y el calculo de quién quedó
"HIRED", y con eso lo cargo en un mini Data Warehouse con el esquema estrella en SQLite.
Al final saco los KPIs y gráficos consultando el DW, no el CSV.

Todo se guarda en Drive: `Mi unidad > Colab Notebooks > Workshop01`.


## Paso 0: Montar Drive y cargar librerías

In [1]:
from google.colab import drive
import os
import pandas as pd
import sqlite3


drive.mount('/content/drive')

# esta es la carpeta donde tengo todo el workshop
base_path = "/content/drive/My Drive/Colab Notebooks/workshop01"


CSV_PATH = os.path.join(base_path, "candidates.csv")
DW_PATH = os.path.join(base_path, "candidates_dw.db")


Mounted at /content/drive


## Paso 1: Extract

Nada raro aca, solo leo el csv tal cual esta, sin tocar nada todavia.


In [2]:
def extract(csv_path):
    # lee el csv y ya, esto es el "E" del ETL
    df = pd.read_csv(csv_path)
    return df


df_raw = extract(CSV_PATH)
print("filas leidas:", len(df_raw))
df_raw.head()


filas leidas: 50000


,First Name,Last Name,Email,Country,Application Date,Yoe,Seniority,Technology,Code Challenge Score,Technical Interview
0,Danielle,Levy,ashleysellers@gmail.com,UK,2024-11-23,6.0,Manager,Mulesoft,0,0.0
1,Angel,Ortega,gary31@yahoo.com,USA,2025-03-21,19.0,NaN,Mulesoft,9,6.0
2,Joshua,Lopez,austingarcia@hotmail.com,Ecuador,2024-10-12,14.0,NaN,Mulesoft,2,5.0
3,Jeffrey,Powers,courtney32@gmail.com,Colombia,2024-10-29,10.0,Lead,Java Backend,3,NaN
4,Jill,Robinson,stokessuzanne@gmail.com,Australia,2022-04-05,7.0,Junior,QA Manual,3,3.0


In [3]:
# reviso rapido como viene la data: tipos y nulos
df_raw.info()
print()
print(df_raw.isna().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   First Name            50000 non-null  object 
 1   Last Name             50000 non-null  object 
 2   Email                 47491 non-null  object 
 3   Country               50000 non-null  object 
 4   Application Date      50000 non-null  object 
 5   Yoe                   47509 non-null  float64
 6   Seniority             47489 non-null  object 
 7   Technology            50000 non-null  object 
 8   Code Challenge Score  50000 non-null  int64  
 9   Technical Interview   47494 non-null  float64
dtypes: float64(2), int64(1), object(7)
memory usage: 3.8+ MB

First Name                 0
Last Name                  0
Email                   2509
Country                    0
Application Date           0
Yoe                     2491
Seniority               2511
Technology   

## Paso 2: Transform

Dos cosas: 1) limpiar la data con los nombres de columnas, tipos, nulos y duplicados., 2) calcular quien quedo HIRED. `Code Challenge Score >= 7` y `Technical Interview >= 7`.


In [4]:
def clean_data(df):
    # esta funcion deja la data lista para trabajar: nombres de columnas
    # ordenados, tipos correctos, nulos rellenos y sin duplicados
    df = df.copy()

    # nombres de columnas
    df.columns = [
        "first_name", "last_name", "email", "country", "application_date",
        "yoe", "seniority", "technology", "code_challenge_score",
        "technical_interview"]

    df = df.drop_duplicates()  # por si hay filas repetidas, todas las columnas deben ser exactamente iguales, es decir un ejemplo, si hay
                               # alguien con los mismos datos repetidos pero con otra fecha no se va a borrar porque son distintos.

    # Convierte la columna a formato de fecha. Si encuentra texto o basura, lo deja como nulo (NaT)
    df["application_date"] = pd.to_datetime(df["application_date"], errors="coerce")

    # los puntajes van de 0 a 10. Encontre filas con "100" en Code Challenge Score,
    # y el unico valor fuera de rango que aparece es justo ese (100) -> asumo que es un
    # error de escritura (les sobra un cero) y lo corrijo a 10, no lo voy a descartar
    df.loc[df["code_challenge_score"] == 100, "code_challenge_score"] = 10
    df.loc[df["technical_interview"] == 10, "technical_interview"] = 10

    # por si acaso llegara algun puntaje negativo, ese si no tiene una correccion obvia,
    # asi que lo trato como dato faltante (0)
    df.loc[df["code_challenge_score"] < 0, "code_challenge_score"] = 0
    df.loc[df["technical_interview"] < 0, "technical_interview"] = 0

    # si falta el puntaje, lo dejo en 0 -> asi nunca cuenta como HIRED por error
    df["code_challenge_score"] = pd.to_numeric(df["code_challenge_score"], errors="coerce").fillna(0)
    df["technical_interview"] = pd.to_numeric(df["technical_interview"], errors="coerce").fillna(0)
    df["yoe"] = pd.to_numeric(df["yoe"], errors="coerce").fillna(0)

    # si falta el texto, mejor dejarlo como "Unknown" que borrar el candidato, se pierde información si borrara todo.
    df["seniority"] = df["seniority"].fillna("Unknown")
    df["country"] = df["country"].fillna("Unknown")
    df["technology"] = df["technology"].fillna("Unknown")
    df["email"] = df["email"].fillna("unknown@unknown.com")

    # esto si lo elimino: sin fecha no puedo ubicarlo en la dimension tiempo
    df = df.dropna(subset=["application_date"])

    return df


df_clean = clean_data(df_raw)
print("filas despues de limpiar:", len(df_clean))
df_clean.head()


filas despues de limpiar: 50000


,first_name,last_name,email,country,application_date,yoe,seniority,technology,code_challenge_score,technical_interview
0,Danielle,Levy,ashleysellers@gmail.com,UK,2024-11-23,6.0,Manager,Mulesoft,0,0.0
1,Angel,Ortega,gary31@yahoo.com,USA,2025-03-21,19.0,Unknown,Mulesoft,9,6.0
2,Joshua,Lopez,austingarcia@hotmail.com,Ecuador,2024-10-12,14.0,Unknown,Mulesoft,2,5.0
3,Jeffrey,Powers,courtney32@gmail.com,Colombia,2024-10-29,10.0,Lead,Java Backend,3,0.0
4,Jill,Robinson,stokessuzanne@gmail.com,Australia,2022-04-05,7.0,Junior,QA Manual,3,3.0


In [5]:
def apply_hired_rule(df):
    # regla del workshop: HIRED si AMBOS puntajes son >= 7
    df = df.copy()
    df["hired"] = (
        (df["code_challenge_score"] >= 7) & (df["technical_interview"] >= 7)
    ).astype(int)
    return df


df_transformed = apply_hired_rule(df_clean)
print("total hired:", df_transformed["hired"].sum())
df_transformed.head()


total hired: 6599


,first_name,last_name,email,country,application_date,yoe,seniority,technology,code_challenge_score,technical_interview,hired
0,Danielle,Levy,ashleysellers@gmail.com,UK,2024-11-23,6.0,Manager,Mulesoft,0,0.0,0
1,Angel,Ortega,gary31@yahoo.com,USA,2025-03-21,19.0,Unknown,Mulesoft,9,6.0,0
2,Joshua,Lopez,austingarcia@hotmail.com,Ecuador,2024-10-12,14.0,Unknown,Mulesoft,2,5.0,0
3,Jeffrey,Powers,courtney32@gmail.com,Colombia,2024-10-29,10.0,Lead,Java Backend,3,0.0,0
4,Jill,Robinson,stokessuzanne@gmail.com,Australia,2022-04-05,7.0,Junior,QA Manual,3,3.0,0


## Paso 3: Armar el esquema estrella

Antes de guardar en el DW, separo la data en dimensiones
y la tabla de hechos (fact_application).

También dejé `dim_application_date` solo con `year`, `month`, `day`
(sin `quarter`, ya no lo necesito para los KPIs).

Dimensiones: `dim_candidate`, `dim_seniority`, `dim_technology`,
`dim_application_date`. Tabla de hechos: `fact_application`.

In [6]:
def build_dimensions(df):
    # dim_candidate: cada fila ya es un candidato unico, asi que el id
    # es simplemente la posicion (empezando en 1)
    dim_candidate = df[["first_name", "last_name", "email", "country"]].reset_index(drop=True)
    dim_candidate.insert(0, "id_candidate", dim_candidate.index + 1)

    # estas si se repiten mucho, por eso les saco los duplicados primero
    dim_seniority = df[["seniority"]].drop_duplicates().reset_index(drop=True)
    dim_seniority.insert(0, "id_seniority", dim_seniority.index + 1)

    dim_technology = df[["technology"]].drop_duplicates().reset_index(drop=True)
    dim_technology.insert(0, "id_technology", dim_technology.index + 1)

    # dim_application_date: solo year, month, day (ya no uso quarter)
    dim_application_date = df[["application_date"]].drop_duplicates().reset_index(drop=True)
    dim_application_date["year"] = dim_application_date["application_date"].dt.year
    dim_application_date["month"] = dim_application_date["application_date"].dt.month
    dim_application_date["day"] = dim_application_date["application_date"].dt.day
    dim_application_date.insert(0, "id_date", dim_application_date.index + 1)

    return {
        "dim_candidate": dim_candidate,
        "dim_seniority": dim_seniority,
        "dim_technology": dim_technology,
        "dim_application_date": dim_application_date,
    }


dims = build_dimensions(df_transformed)
for nombre, d in dims.items():
    print(nombre, "->", len(d), "filas")


dim_candidate -> 50000 filas
dim_seniority -> 8 filas
dim_technology -> 12 filas
dim_application_date -> 1827 filas


In [7]:
def build_fact_table(df, dims):
    # el id_candidate va en el mismo orden que en dim_candidate,
    # asi que no necesito merge para esa parte, solo lo asigno igual
    fact = df.reset_index(drop=True).copy()
    fact.insert(0, "id_candidate", fact.index + 1)

    # estos si los pego por merge, porque son valores repetidos (many-to-one)
    fact = fact.merge(dims["dim_seniority"], on="seniority", how="left")
    fact = fact.merge(dims["dim_technology"], on="technology", how="left")
    fact = fact.merge(
        dims["dim_application_date"][["id_date", "application_date"]],
        on="application_date", how="left"
    )

    # me quedo solo con los ids + las metricas
    fact = fact[[
        "id_candidate", "id_seniority", "id_technology", "id_date",
        "yoe", "code_challenge_score", "technical_interview", "hired"
    ]]

    return fact


fact_application = build_fact_table(df_transformed, dims)
print("filas en fact_application:", len(fact_application))
fact_application.head()


filas en fact_application: 50000


,id_candidate,id_seniority,id_technology,id_date,yoe,code_challenge_score,technical_interview,hired
0,1,1,1,1,6.0,0,0.0,0
1,2,2,1,2,19.0,9,6.0,0
2,3,2,1,3,14.0,2,5.0,0
3,4,3,2,4,10.0,3,0.0,0
4,5,4,3,5,7.0,3,3.0,0


## Paso 4: Load — guardar todo en SQLite (el DW)

Creo la base de datos SQLite con las tablas del
esquema estrella y cargo los DataFrames.


In [8]:
def load_to_dw(dims, fact, dw_path):
    # aca creo las tablas del DW y las lleno con los dataframes de arriba
    conn = sqlite3.connect(dw_path)
    cursor = conn.cursor()

    cursor.executescript("""
    DROP TABLE IF EXISTS fact_application;
    DROP TABLE IF EXISTS dim_candidate;
    DROP TABLE IF EXISTS dim_seniority;
    DROP TABLE IF EXISTS dim_technology;
    DROP TABLE IF EXISTS dim_application_date;

    CREATE TABLE dim_candidate (
        id_candidate INTEGER PRIMARY KEY,
        first_name TEXT,
        last_name TEXT,
        email TEXT,
        country TEXT
    );

    CREATE TABLE dim_seniority (
        id_seniority INTEGER PRIMARY KEY,
        seniority TEXT
    );

    CREATE TABLE dim_technology (
        id_technology INTEGER PRIMARY KEY,
        technology TEXT
    );

    CREATE TABLE dim_application_date (
        id_date INTEGER PRIMARY KEY,
        year INTEGER,
        month INTEGER,
        day INTEGER
    );

    CREATE TABLE fact_application (
        id_candidate INTEGER PRIMARY KEY,
        id_seniority INTEGER,
        id_technology INTEGER,
        id_date INTEGER,
        yoe REAL,
        code_challenge_score REAL,
        technical_interview REAL,
        hired INTEGER,
        FOREIGN KEY (id_candidate) REFERENCES dim_candidate(id_candidate),
        FOREIGN KEY (id_seniority) REFERENCES dim_seniority(id_seniority),
        FOREIGN KEY (id_technology) REFERENCES dim_technology(id_technology),
        FOREIGN KEY (id_date) REFERENCES dim_application_date(id_date)
    );
    """)
    conn.commit()

    # to_sql con if_exists="append" porque las tablas ya las cree arriba
    dims["dim_candidate"].to_sql("dim_candidate", conn, if_exists="append", index=False)
    dims["dim_seniority"].to_sql("dim_seniority", conn, if_exists="append", index=False)
    dims["dim_technology"].to_sql("dim_technology", conn, if_exists="append", index=False)
    dims["dim_application_date"][["id_date", "year", "month", "day"]].to_sql(
        "dim_application_date", conn, if_exists="append", index=False
    )
    fact.to_sql("fact_application", conn, if_exists="append", index=False)

    conn.commit()
    conn.close()


load_to_dw(dims, fact_application, DW_PATH)
print("DW guardado en:", DW_PATH)


DW guardado en: /content/drive/My Drive/Colab Notebooks/workshop01/candidates_dw.db


In [9]:
# chequeo rapido: cuento filas de cada tabla directo desde el DW (no desde los dataframes)
conn = sqlite3.connect(DW_PATH)
for tabla in ["dim_candidate", "dim_seniority", "dim_technology", "dim_application_date", "fact_application"]:
    total = pd.read_sql(f"SELECT COUNT(*) as total FROM {tabla}", conn)["total"][0]
    print(tabla, "->", total, "filas")
conn.close()


dim_candidate -> 50000 filas
dim_seniority -> 8 filas
dim_technology -> 12 filas
dim_application_date -> 1827 filas
fact_application -> 50000 filas
